In [ ]:
from utils import load_mols, load_tokens

root_path = "/cluster/project/jorner/schmiste/flexflow/chemflow/outputs/2026-04-20/09-12-20"
#root_path = "/cluster/project/jorner/schmiste/flexflow/chemflow/outputs/2026-04-16/18-54-14"
token_path = "/cluster/project/jorner/schmiste/flexflow/chemflow/data/qm9/processed"

mols = load_mols(root_path, "invalid_mols.pt")
atom_tokens, edge_tokens, charge_tokens, distributions = load_tokens(token_path)

coordinate_std = distributions["coordinate_std"]

/cluster/project/jorner/schmiste/flexflow/chemflow/.venv/lib/python3.10/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-cluster'. Disabling its usage. Stacktrace: module 'torch_cluster' has no attribute 'knn'
  import torch_geometric.typing


In [2]:
from utils import process_mol

trajs = [
    [process_mol(i, atom_tokens, charge_tokens, edge_tokens) for i in mol_traj]
    for mol_traj in mols
]

In [28]:
from utils import visualize_single_mol

# --- Usage ---
traj_idx = 0

# To visualize just the *last* frame (the final generated molecule) of that trajectory:
final_mol_data = trajs[traj_idx][-1]

view = visualize_single_mol(final_mol_data)
#view.show()


In [29]:
from utils import visualize_variable_topology

# visualize_variable_topology(trajs[traj_idx], interval=50)


In [30]:
from utils import visualize_variable_topology_slider
visualize_variable_topology_slider(trajs[traj_idx])

In [31]:
# import torch
# from collections import Counter
# from pathlib import Path
# from rdkit import Chem
# from rdkit.Chem.Scaffolds import MurckoScaffold
# import pandas as pd

# data_dir = Path("/cluster/project/jorner/schmiste/flexflow/chemflow/data/qm9/processed")  # adjust if needed

# results = {}
# for split in ["train", "val", "test"]:
#     data, _, _ = torch.load(data_dir / f"{split}.pt", weights_only=False)
#     smiles_list = data["smiles"]  # list[str]

#     sg = torch.load(data_dir / f"scaffold_groups_{split}.pt", weights_only=False)
#     mol_to_group = sg["mol_to_group"]  # LongTensor [N], -1 = no scaffold
#     groups = sg["groups"]             # list[list[int]]

#     # Derive scaffold SMILES from the first member of each group
#     group_to_scaffold = {}
#     for gid, idxs in enumerate(groups):
#         mol = Chem.MolFromSmiles(smiles_list[idxs[0]])
#         group_to_scaffold[gid] = Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(mol))

#     scaffold_counts = Counter({group_to_scaffold[gid]: len(idxs) for gid, idxs in enumerate(groups)})
#     n_scaffold = int((mol_to_group >= 0).sum())
#     results[split] = scaffold_counts
#     print(f"{split}: {len(smiles_list)} total, {n_scaffold} with scaffold, {len(groups)} unique scaffolds")

# # Top scaffolds table
# all_scaffolds = set().union(*(r.keys() for r in results.values()))
# df = pd.DataFrame(
#     {split: {s: results[split].get(s, 0) for s in all_scaffolds} for split in ["train", "val", "test"]}
# ).astype(int)
# df["total"] = df.sum(axis=1)
# df.sort_values("val", ascending=False).head(20)